<a href="https://colab.research.google.com/github/fourmansyah/Mapperatorinator/blob/main/colab/mapperatorinator_inference_with_video_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beatmap Generation with Mapperatorinator [MOD]

This notebook is an better interactive demo of an osu! beatmap generation model created by OliBomby [MOD]. This model is capable of generating hit objects, hitsounds, timing, kiai times, and SVs for all 4 gamemodes and can do batch osu file. You can upload a beatmap to give to the model as additional context or remap parts of the beatmap.

### Instructions for running:

* Read and accept the rules regarding using this tool by clicking the checkbox.
* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* __Execute each cell in order__. Press ▶️ on the left of each cell to execute the cell.
* __Setup Environment__: run the first cell to clone the repository and install the required dependencies. You only need to run this cell once per session.
* __Upload Audio__: choose a .mp3 or .ogg file from your computer.
* __Upload Beatmap__: optionally choose a beatmap .osu file from your computer.  You can find these files in stable by using File > Open Song Folder, or in lazer by using File > Edit Externally.
* __Configure__: choose your generation parameters to control the style of the generated beatmap.
* Generate the beatmap using the __Generate Beatmap__ cell. (it may take a few minutes depending on the length of the song)


In [ ]:
#@title 🚀 Setup Environment { display-mode: "form" }
#@markdown ### Use this tool responsibly. Always disclose the use of AI in your beatmaps. Accept the rules and run this cell.
i_accept_the_rules = False # @param {type:"boolean"}

assert i_accept_the_rules, "Read and accept the rules first!"

import os
import importlib.metadata
from packaging.version import Version

if not os.path.exists("/content/Mapperatorinator"):
    !git clone https://github.com/fourmansyah/Mapperatorinator.git

%cd /content/Mapperatorinator

# UPDATE KRUSIAL: Sinkronisasi dengan Official V32 OliBomby
!pip install transformers==4.57.3
!pip install hydra-core nnaudio
!pip install slider git+https://github.com/OliBomby/slider.git
!pip install rosu-pp-py==3.1.0
!pip install peft==0.18.1

# Ekstra tool bawaan milikmu
!pip install yt-dlp
!pip install -U torchao packaging

installed_torchao = importlib.metadata.version("torchao")
print("Installed torchao version:", installed_torchao)

if Version(installed_torchao) < Version("0.16.0"):
    print("⚠️ Incompatible torchao detected. Restarting runtime automatically...")
    os.kill(os.getpid(), 9)

print("✅ Environment ready.")

from google.colab import files
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main

output_path = "output"
input_audio = ""
input_beatmap = ""

In [ ]:
#@title Local Upload Audio { display-mode: "form" }
#@markdown Run this cell to upload audio. This is the song to generate a beatmap for. Please upload a .mp3 or .ogg file.

def upload_audio():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.mp3') and not file.endswith('.ogg'):
        print('Invalid file format. Please upload a .mp3 or .ogg file.')
        return ""
    return data[0]

input_audio = upload_audio()

In [ ]:
#@title 🎵 Download Audio via URL { display-mode: "form" }
#@markdown Enter the YouTube URL or direct link to the audio file. The system will automatically download and display the music player.
audio_url = "" #@param {type:"string"}

import os
import subprocess
from IPython.display import Audio, display

def download_and_validate_audio(url):
    """Download audio from URL with proper error handling and audio player"""
    if not url or url.strip() == "":
        print("❌ URL kosong! Harap masukkan link yang valid.")
        return None

    print(f"⏳ Mendownload audio dari: {url} ...")

    # Nama file bersih agar tidak ada karakter aneh dari judul YouTube
    clean_filename = "audio"
    target_path = os.path.abspath(f"{clean_filename}.mp3")

    # Hapus file lama agar tidak tertumpuk jika user generate lagu kedua
    for ext in ['.mp3', '.ogg', '.wav', '.m4a', '.webm']:
        old_file = os.path.abspath(f"{clean_filename}{ext}")
        if os.path.exists(old_file):
            os.remove(old_file)

    try:
        # Perintah yt-dlp untuk ekstrak audio MP3
        command = [
            "yt-dlp",
            "-x",
            "--audio-format", "mp3",
            "--audio-quality", "0",
            "-o", f"{clean_filename}.%(ext)s",
            url
        ]

        # Menjalankan perintah secara diam-diam agar tampilan Colab tetap rapi
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

        if os.path.exists(target_path):
            print(f"✅ Audio berhasil didownload!")
            print(f"📁 Path: {target_path}")

            # Memunculkan audio player
            display(Audio(target_path))
            return target_path
        else:
            print("❌ File gagal ditemukan setelah proses download selesai.")
            return None

    except subprocess.CalledProcessError:
        print("❌ Terjadi kesalahan saat mendownload. Pastikan URL valid dan video tidak diprivat/dibatasi usia.")
        return None
    except Exception as e:
        print(f"❌ Error sistem: {e}")
        return None

# Eksekusi fungsi download
audio_path = download_and_validate_audio(audio_url)

# Meneruskan variabel ke input_audio agar Segmen 3 (AI Generator) bisa membacanya
if audio_path:
    input_audio = audio_path
else:
    print("\n⚠️ Silakan periksa kembali URL kamu dan jalankan ulang cell ini.")

In [ ]:
#@title (Optional) Upload Beatmap { display-mode: "form" }
#@markdown This step is required if you want to use `in_context` or `add_to_beatmap` to provide additional info to the model.
#@markdown It will also fill in any missing metadata and unknown values in the configuration using info of the reference beatmap.
#@markdown Please upload a **.osu** file. You can find the .osu file in the song folder in stable or by using File > Edit Externally in lazer.
use_reference_beatmap = False # @param {type:"boolean"}

def upload_beatmap():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.osu'):
        print('Invalid file format. Please upload a .osu file.\nIn stable you can find the .osu file in the song folder (File > Open Song Folder).\nIn lazer you can find the .osu file by using File > Edit Externally.')
        return ""
    return file

if use_reference_beatmap:
    input_beatmap = upload_beatmap()
else:
    input_beatmap = ""

In [ ]:
#@title 🎥 Optional Video Manager { display-mode: "form" }
video_mode = "Disable"  # @param ["Disable", "Download + Inject", "Inject Local Video Only", "Download Only"]
#@markdown - Disable = if you don't want to add video
#@markdown - Download + Inject = if you want to add videos to beatmaps and download the resulting videos.
#@markdown - Inject Local Video Only = if you just want to add videos to beatmaps.
#@markdown - Download Only = if you only want to download the video results.

# Isi salah satu sesuai mode:
#@markdown Youtube Share URL
youtube_video_url = ""  # @param {type:"string"}
#@markdown Colab / Google Drive Path [/content/drive/MyDrive/VideoName.mp4]
video_file_path = ""    # @param {type:"string"}

#@markdown #### Video Setting
video_offset = 0  # @param {type:"integer"}
video_target_height = 720  # @param [480, 720] {type:"raw"}
video_crf = 23  # @param {type:"slider", min:18, max:30, step:1}
video_preset = "medium"  # @param ["veryfast", "fast", "medium", "slow"]
video_output_name = "beatmap_video"  # @param {type:"string"}

import os
import re
import shutil
import subprocess
from google.colab import files

prepared_video_path = None
video_should_inject = False

def _run(cmd):
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True)

def _safe_name(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]+', "_", name).strip()
    return name or "beatmap_video"

def inject_video_event(osu_path, video_filename, video_offset=0):
    if not video_filename:
        return

    video_basename = os.path.basename(video_filename)

    with open(osu_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    events_index = None
    for i, line in enumerate(lines):
        if line.strip() == "[Events]":
            events_index = i
            break

    if events_index is None:
        print(f"[WARN] [Events] section tidak ditemukan di {osu_path}")
        return

    for line in lines[events_index + 1:]:
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            break
        if stripped.startswith("Video,") or stripped.startswith("1,"):
            print(f"[INFO] Video event sudah ada di {osu_path}, skip.")
            return

    insert_index = events_index + 1
    while insert_index < len(lines):
        stripped = lines[insert_index].strip()
        if stripped.startswith("//") or stripped == "":
            insert_index += 1
        else:
            break

    video_line = f'Video,{video_offset},"{video_basename}",0,0\n'
    lines.insert(insert_index, video_line)

    with open(osu_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print(f"✅ Video injected into: {osu_path}")

if video_mode == "Disable":
    print("ℹ️ Video feature disabled.")

else:
    if shutil.which("yt-dlp") is None and video_mode in ["Download + Inject", "Download Only"]:
        !pip -q install -U yt-dlp
    if shutil.which("ffmpeg") is None:
        !apt -qq update
        !apt -qq install -y ffmpeg

    os.makedirs("/content/video_tmp", exist_ok=True)
    out_name = _safe_name(video_output_name)
    encoded_output = f"/content/{out_name}.mp4"

    if video_mode in ["Download + Inject", "Download Only"]:
        if not youtube_video_url.strip():
            raise ValueError("Isi youtube_video_url untuk mode download.")

        template = "/content/video_tmp/%(title)s.%(ext)s"
        _run([
            "yt-dlp",
            "-f", "bv*+ba/b",
            "--merge-output-format", "mp4",
            "-o", template,
            youtube_video_url.strip()
        ])

        candidates = []
        for root, _, files_ in os.walk("/content/video_tmp"):
            for f in files_:
                if f.lower().endswith((".mp4", ".mkv", ".webm", ".mov")):
                    candidates.append(os.path.join(root, f))

        if not candidates:
            raise FileNotFoundError("Tidak menemukan video hasil download.")

        source_video = max(candidates, key=os.path.getmtime)

        vf = f"scale='if(gt(ih,{video_target_height}),-2,iw)':'if(gt(ih,{video_target_height}),{video_target_height},ih)'"

        _run([
            "ffmpeg", "-y",
            "-i", source_video,
            "-an",
            "-vf", vf,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", video_preset,
            "-crf", str(video_crf),
            encoded_output
        ])

        prepared_video_path = encoded_output
        print("✅ Prepared video:", prepared_video_path)

        if video_mode == "Download Only":
            files.download(prepared_video_path)
            print("⬇️ Video downloaded to your laptop.")
        else:
            video_should_inject = True
            print("🎬 Video ready for injection after beatmap generation.")

    elif video_mode == "Inject Local Video Only":
        if not video_file_path.strip():
            raise ValueError("Isi video_file_path untuk mode local inject.")
        if not os.path.exists(video_file_path.strip()):
            raise FileNotFoundError(f"File tidak ditemukan: {video_file_path}")

        prepared_video_path = video_file_path.strip()
        video_should_inject = True
        print("🎬 Local video ready for injection:", prepared_video_path)


In [ ]:
#@title 🎚️ Configure Preset Generator [V32 Optimized] { display-mode: "form" }

#@markdown ### 1. CHOOSE MODEL & PRESET
#@markdown Select a model (V32 is highly recommended) and a preset difficulty level.
model = "Mapperatorinator V32" # @param ["Mapperatorinator V29", "Mapperatorinator V30", "Mapperatorinator V31", "Mapperatorinator V32-mini", "Mapperatorinator V32"]
gamemode = "standard"  # @param ["standard", "taiko", "catch the beat", "mania"]
diff_preset = "Normal"  # @param ["Normal", "Hard", "Insane", "Expert", "Extra"]

#@markdown ### 2. SONG INFORMATION (Metadata)
#@markdown Fill in the metadata for the .osu file. Leave it blank if you want the system to fill it in.
title = ""  # @param {type:"string"}
artist = ""  # @param {type:"string"}
creator = "XX_li7fg"  # @param {type:"string"}
version_name = ""  # @param {type:"string"}
tags = ""  # @param {type:"string"}
preview_time = -1  # @param {type:"integer"}

#@markdown ### 3. ADVANCED SETTINGS (Opsional)
#@markdown Fill in BPM & Offset manually for more precise mapping results & prevent LoRA V32 errors.
bpm = -1 # @param {type:"number"}
offset = -1 # @param {type:"integer"}
lora_path = "" # @param ["OliBomby/Mapperatorinator-v30-LoRA-Kroytz", "OliBomby/Mapperatorinator-v30-LoRA-2025", "mouceen/Mapperatorinator-v30-LoRA-sliderSlop", "mouceen/Mapperatorinator-v30-LoRA-Arles-v1", "mjoink/Mapperatorinator-v30-LoRA-aim-control", "Nyukul/Mapperatorinator-v30-LoRA-NM1", "Tiger14n/lora_model", "fourmansyah/Mapperatorinator-v32-LoRA-Kroytz", "fourmansyah/Mapperatorinator-v32-LoRA-NM1", "OliBomby/Mapperatorinator-v32-LoRA-Voxell", "fourmansyah/LoRA-Arles1-v32", "fourmansyah/LoRA-2025-v32", "fourmansyah/LoRA-sliderSlop-v32"] {allow-input: true}
seed = -1  # @param {type:"integer"}
quantity = 1  # @param {type:"slider", min:1, max:10, step:1}
download_mode = "Individual .osu Files"  # @param ["Master .osz (Lagu + Semua Map)", "Individual .osu Files"]

# [Kode Python di bawah sini tetap sama seperti versi yang baru saja kita perbaiki sebelumnya]
# =========================
# IMPORT
# =========================
import os
import time
import random
import zipfile
import shutil
import traceback
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main
from google.colab import files

# =========================
# PRESET TERPISAH (V32 Descriptor Ready)
# =========================
PRESETS = {
    "Normal": {
        "difficulty": 2.2, "hp_drain_rate": 3.5, "circle_size": 4, "overall_difficulty": 5.5,
        "approach_rate": 6.5, "slider_multiplier": 1.6, "slider_tick_rate": 1,
        "descriptors": ["style/clean", "expression/simple", "streams/flow aim"],
        "negative_descriptors": ["expression/chaotic", "reading/visually dense", "skillset/tech"],
        "cfg_scale": 1.4, "temperature": 0.96, "top_p": 0.95, "top_k": 0, "num_beams": 1,
        "do_sample": True, "timeshift_bias": 0.0, "timing_leniency": 22, "timing_temperature": 0.10,
        "timer_cfg_scale": 1.0, "timer_num_beams": 2, "timer_iterations": 20, "timer_bpm_threshold": 0.1,
        "super_timing": False, "bpm": None, "offset": None, "resnap_events": True,
        "diff_cfg_scale": 1.0, "refine_iters": 8, "random_init": False, "lookback": 0.5, "lookahead": 0.4,
    },
    "Hard": {
        "difficulty": 3.6, "hp_drain_rate": 4.5, "circle_size": 4, "overall_difficulty": 6.8,
        "approach_rate": 8.0, "slider_multiplier": 1.5, "slider_tick_rate": 1,
        "descriptors": ["style/clean", "streams/flow aim", "streams/bursts"],
        "negative_descriptors": ["expression/chaotic", "reading/visually dense", "reading/overlaps"],
        "cfg_scale": 1.6, "temperature": 0.94, "top_p": 0.95, "top_k": 0, "num_beams": 1,
        "do_sample": True, "timeshift_bias": 0.0, "timing_leniency": 20, "timing_temperature": 0.09,
        "timer_cfg_scale": 1.0, "timer_num_beams": 2, "timer_iterations": 20, "timer_bpm_threshold": 0.1,
        "super_timing": False, "bpm": None, "offset": None, "resnap_events": True,
        "diff_cfg_scale": 1.1, "refine_iters": 10, "random_init": False, "lookback": 0.5, "lookahead": 0.4,
    },
    "Insane": {
        "difficulty": 4.8, "hp_drain_rate": 5.5, "circle_size": 4, "overall_difficulty": 8.0,
        "approach_rate": 9.0, "slider_multiplier": 1.4, "slider_tick_rate": 1,
        "descriptors": ["style/clean", "skillset/jumps", "streams/flow aim"],
        "negative_descriptors": ["style/messy", "expression/chaotic", "reading/visually dense"],
        "cfg_scale": 1.8, "temperature": 0.92, "top_p": 0.95, "top_k": 0, "num_beams": 1,
        "do_sample": True, "timeshift_bias": 0.0, "timing_leniency": 18, "timing_temperature": 0.08,
        "timer_cfg_scale": 1.0, "timer_num_beams": 2, "timer_iterations": 20, "timer_bpm_threshold": 0.1,
        "super_timing": False, "bpm": None, "offset": None, "resnap_events": True,
        "diff_cfg_scale": 1.2, "refine_iters": 12, "random_init": False, "lookback": 0.5, "lookahead": 0.4,
    },
    "Expert": {
        "difficulty": 5.6, "hp_drain_rate": 6.0, "circle_size": 4, "overall_difficulty": 8.7,
        "approach_rate": 9.4, "slider_multiplier": 1.4, "slider_tick_rate": 1,
        "descriptors": ["skillset/jumps", "skillset/precision", "streams/bursts"],
        "negative_descriptors": ["style/messy", "expression/chaotic", "reading/overlaps"],
        "cfg_scale": 2.0, "temperature": 0.90, "top_p": 0.94, "top_k": 0, "num_beams": 1,
        "do_sample": True, "timeshift_bias": 0.0, "timing_leniency": 16, "timing_temperature": 0.08,
        "timer_cfg_scale": 1.1, "timer_num_beams": 2, "timer_iterations": 20, "timer_bpm_threshold": 0.1,
        "super_timing": False, "bpm": None, "offset": None, "resnap_events": True,
        "diff_cfg_scale": 1.25, "refine_iters": 14, "random_init": False, "lookback": 0.5, "lookahead": 0.4,
    },
    "Extra": {
        "difficulty": 6.4, "hp_drain_rate": 6.5, "circle_size": 4, "overall_difficulty": 9.2,
        "approach_rate": 9.7, "slider_multiplier": 1.3, "slider_tick_rate": 1,
        "descriptors": ["skillset/jumps", "skillset/precision", "skillset/alt"],
        "negative_descriptors": ["style/messy", "expression/chaotic", "reading/visually dense"],
        "cfg_scale": 2.2, "temperature": 0.88, "top_p": 0.93, "top_k": 0, "num_beams": 1,
        "do_sample": True, "timeshift_bias": 0.0, "timing_leniency": 15, "timing_temperature": 0.07,
        "timer_cfg_scale": 1.1, "timer_num_beams": 2, "timer_iterations": 20, "timer_bpm_threshold": 0.1,
        "super_timing": False, "bpm": None, "offset": None, "resnap_events": True,
        "diff_cfg_scale": 1.3, "refine_iters": 16, "random_init": False, "lookback": 0.55, "lookahead": 0.45,
    },
}

preset = PRESETS[diff_preset]

# =========================
# KONVERSI PARAMETER
# =========================
a_config = model.split(" ")[-1].lower()
a_gamemode = ["standard", "taiko", "catch the beat", "mania"].index(gamemode)

a_title = None if title == "" else title
a_artist = None if artist == "" else artist
a_creator = None if creator == "" else creator
a_tags = None if tags == "" else tags
a_background = None if background == "" else background
a_preview_time = None if preview_time == -1 else preview_time
a_mapper_id = None if mapper_id == -1 else mapper_id
a_beatmap_id = None if beatmap_id == -1 else beatmap_id
a_year = None if year == -1 else year
a_lora_path = None if lora_path == "" else lora_path
a_start_time = None if start_time == -1 else start_time
a_end_time = None if end_time == -1 else end_time
a_in_context = [ContextType(c.lower()) for c in in_context[1:-1].split(",")]
a_output_type = [ContextType(c.lower()) for c in output_type[1:-1].split(",")]
a_seed = None if seed == -1 else seed

# Tangkap opsi BPM & Offset
a_bpm = None if bpm == -1 else bpm
a_offset = None if offset == -1 else offset

# pakai nama preset sebagai version kalau kosong
a_base_version = version_name if version_name.strip() else diff_preset

# =========================
# VALIDASI LORA & MODEL
# =========================
if a_lora_path:
    lora_lower = a_lora_path.lower()
    if "v32" in a_config and "v30" in lora_lower:
        print("⚠️ PERINGATAN KRITIS: Kamu memilih Model V32, tapi memasukkan LoRA V30! Mematikan LoRA...")
        a_lora_path = None
    elif "v30" in a_config and "v32" in lora_lower:
        print("⚠️ PERINGATAN KRITIS: Kamu memilih Model V30, tapi memasukkan LoRA V32! Mematikan LoRA...")
        a_lora_path = None

    if a_lora_path and "v32" in a_config and a_bpm is None:
        print("💡 INFO LORA V32: BPM kosong (-1). Tulisan 'Skipping LoRA' mungkin muncul nanti, JANGAN PANIK, itu normal!")

# Validasi Basic
if any(c in a_in_context for c in [ContextType.TIMING, ContextType.KIAI, ContextType.MAP, ContextType.SV, ContextType.GD, ContextType.NO_HS]) or add_to_beatmap:
    assert os.path.exists(input_beatmap), "Please upload a reference beatmap."

assert os.path.exists(input_audio), "Please upload an audio file."

if "v30" in a_config:
    assert a_gamemode == 0, "V30 only supports standard mode."
    if any(c in a_in_context for c in [ContextType.KIAI, ContextType.MAP, ContextType.SV]):
        print("WARNING: V30 does not support KIAI, MAP, or SV in_context, ignoring.")
    if output_type != "[MAP]":
        print("WARNING: V30 only supports [MAP] output type, setting output type to [MAP].")
        a_output_type = [ContextType.MAP]
    if preset["super_timing"]:
        print("WARNING: V30 does not fully support super timing, generation will be VERY slow.")

# =========================
# SIAPKAN OUTPUT
# =========================
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path, exist_ok=True)

# --- Jembatan Video Otomatis (Sebelum Loop) ---
if 'video_should_inject' in globals() and video_should_inject:
    if 'prepared_video_path' in globals() and prepared_video_path and os.path.exists(prepared_video_path):
        video_dest = os.path.join(output_path, os.path.basename(prepared_video_path))
        shutil.copy(prepared_video_path, video_dest)
        print(f"📁 Video sudah siap di folder output.")
# --------------------------------------

generated_osu_files = []

print(f"🚀 Generating {quantity} beatmap(s) using preset: {diff_preset}")
print("==================================================")

# =========================
# LOOP GENERATE
# =========================
for i in range(quantity):
    take_num = i + 1
    current_version = f"{a_base_version} (Take {take_num})" if quantity > 1 else a_base_version
    current_seed = random.randint(1, 999999) if a_seed is None else (a_seed + i)

    print(f"\n🎬 Take {take_num}/{quantity} | Diff: {current_version} | Seed: {current_seed}")

    try:
        try:
            from hydra.core.global_hydra import GlobalHydra
            GlobalHydra.instance().clear()
        except:
            pass

        with initialize_config_dir(version_base="1.1", config_dir="/content/Mapperatorinator/configs/inference"):
            conf = compose(config_name=a_config)

        # path
        conf.audio_path = input_audio
        conf.output_path = output_path
        conf.beatmap_path = input_beatmap if 'input_beatmap' in globals() else None

        # metadata
        conf.gamemode = a_gamemode
        conf.title = a_title
        conf.artist = a_artist
        conf.creator = a_creator
        conf.version = current_version
        conf.tags = a_tags
        conf.background = a_background
        conf.preview_time = a_preview_time

        # optional user overrides
        conf.mapper_id = a_mapper_id
        conf.beatmap_id = a_beatmap_id
        conf.year = a_year
        conf.lora_path = a_lora_path
        conf.hitsounded = hitsounded
        conf.generate_positions = generate_positions

        # preset difficulty block
        conf.difficulty = preset["difficulty"]
        conf.hp_drain_rate = preset["hp_drain_rate"]
        conf.circle_size = preset["circle_size"]
        conf.overall_difficulty = preset["overall_difficulty"]
        conf.approach_rate = preset["approach_rate"]
        conf.slider_multiplier = preset["slider_multiplier"]
        conf.slider_tick_rate = preset["slider_tick_rate"]

        # preset style
        conf.descriptors = preset["descriptors"]
        conf.negative_descriptors = preset["negative_descriptors"]

        # generation bounds/context
        conf.export_osz = False
        conf.add_to_beatmap = add_to_beatmap
        conf.start_time = a_start_time
        conf.end_time = a_end_time
        conf.in_context = a_in_context
        conf.output_type = a_output_type
        conf.seed = current_seed

        # preset sampling
        conf.cfg_scale = preset["cfg_scale"]
        conf.temperature = preset["temperature"]
        conf.top_p = preset["top_p"]
        conf.top_k = preset["top_k"]
        conf.num_beams = preset["num_beams"]
        conf.do_sample = preset["do_sample"]
        conf.timeshift_bias = preset["timeshift_bias"]

        # preset timing (Memprioritaskan input manual di UI jika diisi)
        conf.timing_leniency = preset["timing_leniency"]
        conf.timing_temperature = preset["timing_temperature"]
        conf.timer_cfg_scale = preset["timer_cfg_scale"]
        conf.timer_num_beams = preset["timer_num_beams"]
        conf.timer_iterations = preset["timer_iterations"]
        conf.timer_bpm_threshold = preset["timer_bpm_threshold"]
        conf.super_timing = preset["super_timing"]
        conf.bpm = a_bpm if a_bpm is not None else preset["bpm"]
        conf.offset = a_offset if a_offset is not None else preset["offset"]
        conf.resnap_events = preset["resnap_events"]

        # preset positions
        conf.diff_cfg_scale = preset["diff_cfg_scale"]
        conf.refine_iters = preset["refine_iters"]
        conf.random_init = preset["random_init"]

        # long-map consistency
        conf.lookback = preset["lookback"]
        conf.lookahead = preset["lookahead"]

        _, result_path, _ = main(conf)

# --- Jembatan Video Otomatis (Hanya injeksi event) ---
        if result_path and result_path.endswith(".osu"):
            if 'video_should_inject' in globals() and video_should_inject:
                inject_video_event(result_path, prepared_video_path, video_offset)
                print(f"🎥 Video berhasil disuntikkan ke: {os.path.basename(result_path)}")
        # ----------------------------------------------------

        if result_path and result_path.endswith(".osu"):
            generated_osu_files.append(result_path)
            print(f"✅ Finished Take {take_num}")

    except Exception as e:
        print(f"❌ Error on Take {take_num}: {e}")
        traceback.print_exc()

# =========================
# PACKAGING
# =========================
if generated_osu_files:
    if download_mode == "Master .osz (Lagu + Semua Map)":
        file_title = a_title if a_title else "Unknown Title"
        file_artist = a_artist if a_artist else "Unknown Artist"
        master_osz_name = f"{file_artist} - {file_title} ({diff_preset} Batch).osz"

        with zipfile.ZipFile(master_osz_name, "w", zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(input_audio, os.path.basename(input_audio))
            if a_background and os.path.exists(a_background):
                zipf.write(a_background, os.path.basename(a_background))
            for osu_file in generated_osu_files:
                zipf.write(osu_file, os.path.basename(osu_file))

        print(f"🎉 Downloading: {master_osz_name}")
        files.download(master_osz_name)

    else:
        for osu_file in generated_osu_files:
            print(f"Downloading: {os.path.basename(osu_file)}")
            files.download(osu_file)
            time.sleep(1.5)
else:
    print("❌ No .osu file generated.")